In [ ]:
import numpy as np
import pandas as pd
import MDAnalysis as mda
import matplotlib.pyplot as plt
import plotly.express as px
import scipy.stats as stats


In [ ]:
total_dist_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/dist_data/{i}/{j}_cent_dists.csv', index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = [np.mean(temp_df['avg_cent_dist'])]
        total_dist_df = pd.concat([total_dist_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
    elif i == '2_spread':
        name = 'GXGGGGGXG'
    elif i == '3_close':
        name = 'GGGXXXGGG'
    else:
        name = 'GXGGXGGXG'
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/dist_data/close_spread/{i}_cent_dists.csv', index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = [np.mean(temp_df['avg_cent_dist'])]
    total_dist_df = pd.concat([total_dist_df, temp_df_2], axis=1)


total_dist_df

In [ ]:
total_size_df = pd.DataFrame()

for i in ['3res','9res']:
    for j in ['equal','charged','aromatic','polar','nonpolar']:
        temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/{i}/{j}_size.csv', index_col=0)
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{i[0]}{j[0].upper()}'] = temp_df['prot-size'].values
        total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)
        
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
        num_stickers = 2
    elif i == '2_spread':
        name = 'GXGGGGGXG'
        num_stickers = 2
    elif i == '3_close':
        name = 'GGGXXXGGG'
        num_stickers = 3
    else:
        name = 'GXGGXGGXG'
        num_stickers = 3
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/size_data/close_spread/{i}_size.csv', index_col=0)
    temp_df_2 = pd.DataFrame()
    temp_df_2[f'{name}'] = temp_df['prot-size'].values / num_stickers
    total_size_df = pd.concat([total_size_df, temp_df_2], axis=1)


total_size_df

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})


In [ ]:
total_asphericity_df = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/asphericity_data.csv', index_col=0)

In [ ]:
time_asphericity_df = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/asphericity_time_data.csv', index_col=0)

In [ ]:
time_asphericity_df

In [ ]:
cleaned_size = [total_size_df[col].dropna() for col in total_size_df.columns]
cleaned_asph = [time_asphericity_df[col].dropna() for col in time_asphericity_df.columns]

In [ ]:
time_asphericity_df.mean(axis=0)

In [ ]:
colors = [
    # Set 2: Greens (lightest to darkest)
    "#2CA02C", "#2CA02C", "#2CA02C", "#2CA02C", "#2CA02C",
    # Set 3: Purples (lightest to darkest)
    '#7E4AB8', '#7E4AB8', '#7E4AB8', '#7E4AB8', '#7E4AB8',
    # Set 4: Oranges (lightest to darkest)
    "#FF7F0E", "#FF7F0E", "#FF7F0E", "#FF7F0E"]

fig, ax1 = plt.subplots(figsize=[7,5])

cleaned_size = [total_size_df[col].dropna() for col in total_size_df.columns]
cleaned_asph = [time_asphericity_df[col].dropna() for col in time_asphericity_df.columns]

violins = ax1.violinplot(cleaned_size, showmeans=False, showmedians=False)

# Customize each violin with colors
for i, body in enumerate(violins['bodies']):
    body.set_facecolor(colors[i])  # Set the fill color
    body.set_edgecolor('none')   # Optional: Set edge color
    body.set_alpha(0.7)           # Optional: Adjust transparency
    
violins['cbars'].set_colors(colors)

violins['cmins'].set_colors(colors)
    
violins['cmaxes'].set_colors(colors)

# Add labels and formatting for the first y-axis
ax1.set_xticks(range(1, len(total_size_df.columns) + 1))
ax1.set_xticklabels(total_size_df.columns, fontsize=12, rotation=45)
ax1.set_ylabel('Clustered Residue Ratio', fontsize=14)
ax1.set_xlabel('System', fontsize=14)
ax1.tick_params(axis='y', labelsize=12)

# Secondary y-axis
ax2 = ax1.twinx()

# Scatter plot data
# scatter_values = total_asphericity_df.iloc[0, :]  # Assuming the values are in the first row
scatter_values = time_asphericity_df.mean(axis=0)  # Assuming the values are in the first row

x_positions = range(1, len(scatter_values) + 1)

# Scatter plot
ax2.scatter(x_positions, scatter_values, color='firebrick', label='Avg Distance from Centroid', zorder=3)
# ax2.violinplot(cleaned_asph, showmeans=True, showmedians=True)

# Add labels and formatting for the second y-axis
ax2.set_ylabel('Average Condensate Asphericity', fontsize=14, color='firebrick')
ax2.tick_params(axis='y', labelsize=12, colors='firebrick')

plt.tight_layout()

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/sizes/size_asphericity_violin_scatter.png', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/sizes/size_asphericity_violin_scatter.pdf', dpi=500, transparent=True)


In [ ]:
water_content = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/good_water_contents/tot_water_content.csv', index_col=0)
water_content.index = water_content.index * (5000/np.max(water_content.index)) 

water_content = water_content.rename(columns={'3res_equal':'3E', '3res_charged':'3C', '3res_aromatic':'3A', '3res_polar':'3P',
       '3res_nonpolar':'3N', '9res_equal':'9E', '9res_charged':'9C', '9res_aromatic':'9A',
       '9res_polar':'9P', '9res_nonpolar':'9N', '2_close':'X2','2_spread':'X2S', '3_close':'X3',
       '3_spread':'X3S'})

total_size_df = total_size_df.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})


In [ ]:
size_water_df = pd.DataFrame(index=water_content.columns, columns=['size','water_content'])

for column in total_size_df.columns:
    size_water_df.loc[f'{column}', 'size'] = np.nanmean(total_size_df[f'{column}'])

for column in water_content.columns:
    size_water_df.loc[f'{column}', 'water_content'] = np.nanmean(water_content[f'{column}'])


In [ ]:
size_water_df['size'] = size_water_df['size'] - 0.01

In [ ]:
# Create a 3D scatter plot
plt.style.use("default")
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111)

# Plot the data points
ax.scatter(size_water_df['size'], size_water_df['water_content'], c='r', marker='o', alpha=0)
                                                    
for i, let in enumerate(size_water_df.index):
    if let in ['3E','3C','3A','3P','3N']:
        color = '#2CA02C'
    elif let in ['9E','9C','9A','9P','9N']:
        color = '#7E4AB8'
    else:
        color = '#FF7F0E'
    ax.text(size_water_df['size'][i], size_water_df['water_content'][i], size_water_df.index[i], fontsize=20, color = color)
                                                    
ax.set_ylabel('Water:Peptide Mass Ratio', fontsize=14)
ax.set_xlabel('Clustered Residue Ratio', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12, pad=0)
plt.ylim(0.3,2.73)
plt.xlim(0.63,1.02)

# name = system.capitalize()

# ax.set_title(f'{name} Stoichiometry')

plt.tight_layout();

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/size_water_scatter.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/size_water_scatter.png', dpi=500, transparent=True)

In [ ]:
hist_colors = ['gray','#b30039','#e3ba14','#7aaa3d','#ac5cad']
kde_colors = ['#51524f','#b30039','#e3ba14','#5a7d2e','#743d75']
labels = ['GGGGXGGGG', 'GGGXGXGGG', 'GXGGGGGXG', 'GGGXXXGGG', 'GXGGXGGXG']

ax = res_place.plot.hist(figsize=[7,5], alpha=0.6, density=True, color=hist_colors)
res_place.plot.kde(ax=ax, figsize=[7,5], alpha=1, color=kde_colors)

ax.set_xlim(0, 1.1)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.set_xlabel("Fraction of Clustered Residues", fontsize=14)
ax.set_ylabel("Density", fontsize=14)
ax.legend(fontsize=12)

for i in [0,1,2,3,4]:
    line = ax.lines[i]
    line.set_linewidth(2) 

patches = [plt.Rectangle((0, 0), 1, 1, color=color, alpha=0.7) for color in hist_colors]
ax.legend(patches, labels, fontsize=12)

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/sizes/res_placement_size_hist+kde.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/sizes/res_placement_size_hist+kde.png', dpi=500, transparent=True)


In [ ]:
import plotly.figure_factory as ff

# df = nonpolar
# name = 'nonpolar'

df_list = [equal_dists,charged_dists,aromatic_dists,polar_dists,nonpolar_dists]
df_name_list = ['equal','charged','aromatic','polar','nonpolar']

hist_data = []

for i in ['ARG']:
    
    for df in df_list:  
    
        hist_data.append(df[f'{i}'][~np.isnan(df[f'{i}'])])
    
    fig = ff.create_distplot(hist_data, group_labels=df_name_list, show_rug=False, show_hist=True)
    
    fig.update_layout(xaxis_title="Distance from Centroid (Angstroms)", yaxis_title="Probability", plot_bgcolor='white')
    fig.update_xaxes(ticks='outside', showline=True, linecolor='black', gridcolor='lightgrey', gridwidth=0.1)
    fig.update_yaxes(ticks='outside', showline=True, linecolor='black', gridcolor='lightgrey', gridwidth=0.1)
    fig.show()
    
#     fig.write_image("images/fig1.png", dpi=500, transparent=True)
    
#     density = stats.gaussian_kde(df[f'{i}'][~np.isnan(df[f'{i}'])])
#     n, x, _ = plt.hist(df[f'{i}'][~np.isnan(df[f'{i}'])], bins=50, 
#                    histtype=u'step', density=True, alpha=0)  
#     plt.plot(x, density(x), label=f'{i}')
    
#     df[f'{i}'].kde(alpha=0.3, bins=50, label=f'{i}')

# plt.legend(ncols=2)
# plt.xlabel('Distance from Centroid (Angstroms)')
# plt.ylabel('Probability')
# plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{name}_dist_dist.png', transparent=True, dpi=500)
# plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{name}_dist_dist.pdf', transparent=True, dpi=500)



In [ ]:
print('equal', (np.mean(equal_size['prot-size'])))
print('charged', (np.mean(charged_size['prot-size'])))
print('aromatic', (np.mean(aromatic_size['prot-size'])))
print('polar', (np.mean(polar_size['prot-size'])))
print('nonpolar', (np.mean(nonpolar_size['prot-size'])))

In [ ]:
step = 20

fig = plt.subplots(figsize=[7,5])
plt.plot(equal_size['frame'][::step]*10000*0.04/10000, equal_size['prot-size'][::step],label = 'equal')
plt.plot(charged_size['frame'][::step]*10000*0.04/10000, charged_size['prot-size'][::step],label = 'charged')
plt.plot(aromatic_size['frame'][::step]*10000*0.04/10000, aromatic_size['prot-size'][::step],label = 'aromatic')
plt.plot(polar_size['frame'][::step]*10000*0.04/10000, polar_size['prot-size'][::step],label = 'polar')
plt.plot(nonpolar_size['frame'][::step]*10000*0.04/10000, nonpolar_size['prot-size'][::step],label = 'nonpolar')
plt.legend(loc='lower center', ncols=2);
